# Session 3: RepWise's MCP server

This notebook calls RepWise's own [`src/mcp_server/server.py`](../src/mcp_server/server.py) over the Model Context Protocol. The server exposes every tool of the single catalog [`config/tools.json`](../config/tools.json), with the same pydantic schemas and database code as the RepWise API and agents.

The server is bound to one account when it starts: it verifies an access token, so **no tool takes a `student_id`** and a model cannot act for another account. Writes that need the student's approval are not run; they come back as `pending_approval`.

It needs no API key and touches no real data: it creates a temporary database with one synthetic account. To run it, use **Run all**, in Colab or from a local checkout.

In [ ]:
# @title Connect to RepWise's MCP server
# Works in Colab (clones the public repository) and from a local checkout.
import asyncio, json, os, secrets, subprocess, sys, tempfile
from pathlib import Path

try:
    import dotenv, fastmcp
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "fastmcp", "python-dotenv"],
        check=True,
    )
from fastmcp import Client

WORK = tempfile.mkdtemp(prefix="repwise-s3-")
try:
    sys.stderr.fileno()  # The MCP stdio client needs a real file for server logs.
except Exception:
    sys.stderr = open(os.path.join(WORK, "mcp_servers.log"), "a")

REPO = next(
    (
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "src" / "mcp_server").exists()
    ),
    None,
)
if REPO is None:
    if not Path("RepWise").exists():
        subprocess.run(
            ["git", "clone", "-q", "https://github.com/Kingflow-23/RepWise.git"],
            check=True,
        )
    REPO = Path("RepWise").resolve()

# A temporary database with one synthetic account, and a token for it signed with
# a throwaway secret, exactly like a token from the API's login route.
DB = os.path.join(WORK, "repwise_demo.sqlite3")
SECRET = secrets.token_hex(32)
seed = subprocess.run(
    [
        sys.executable,
        "-c",
        "import sys; from src.db.demo import seed_demo_account; "
        "from src.api.auth import create_access_token; "
        "student, plan = seed_demo_account(sys.argv[1]); "
        "print(student, plan, create_access_token(student, sys.argv[2]))",
        DB,
        SECRET,
    ],
    cwd=REPO,
    capture_output=True,
    text=True,
    check=True,
)
STUDENT_ID, PLAN_ID, TOKEN = seed.stdout.split()

REPWISE_SERVER = {
    "command": sys.executable,
    "args": ["-m", "src.mcp_server.server"],
    "cwd": str(REPO),
    "env": {
        "FASTMCP_LOG_LEVEL": "WARNING",
        "REPWISE_DATABASE_PATH": DB,
        "REPWISE_LOG_DIR": WORK,
        "REPWISE_AUTH_SECRET": SECRET,
        "REPWISE_MCP_ACCESS_TOKEN": TOKEN,
        "PYTHONIOENCODING": "utf-8",
    },
}
REPWISE_CONFIG = {"mcpServers": {"repwise": REPWISE_SERVER}}


def output_of(result):
    """Return a tool's validated output as a dictionary."""
    return result.structured_content


print("Server: src.mcp_server.server, bound to one synthetic account by its token")
print("Database: temporary")

## Discover the tools

The client asks the server what it offers. The names and descriptions come from `config/tools.json`, and the input schemas from the tools' pydantic models, so `difficulty` becomes an enum. No schema has a `student_id`.

In [ ]:
# @title Discover the tools
async with Client(REPWISE_CONFIG) as mc:
    tools = await mc.list_tools()
for t in tools:
    print(
        f"{t.name}({', '.join(t.input_schema.get('properties', {}))})\n   {t.description}\n"
    )
difficulty = next(t for t in tools if t.name == "save_workout_feedback").input_schema[
    "properties"
]["difficulty"]
print("difficulty enum:", difficulty["enum"])
print(
    "Any tool asks for student_id:",
    any("student_id" in json.dumps(t.input_schema) for t in tools),
)

## Call the tools

This is what a host does when a model asks for a call. The timetable supplies the values the save needs, invalid input is rejected, a consequential change is held for approval, and a new server process reloads the saved feedback.

In [ ]:
# @title Call the tools, without an LLM
async with Client(REPWISE_CONFIG) as mc:
    profile = output_of(await mc.call_tool("get_user_profile", {}))
    print("get_user_profile ->", json.dumps(profile["profile"], indent=1))

    timetable = output_of(await mc.call_tool("get_timetable", {}))
    print(
        "\nget_timetable ->",
        timetable["available_days"],
        timetable["session_minutes"],
        "minutes",
    )
    for w in timetable["planned_workouts"]:
        print(
            f"   {w['day']}: {w['focus']} (week {w['week_number']}, workout_index {w['workout_index']})"
        )

    first = timetable["planned_workouts"][0]
    saved = output_of(
        await mc.call_tool(
            "save_workout_feedback",
            {
                "week_number": first["week_number"],
                "workout_index": first["workout_index"],
                "completion_status": "completed",
                "difficulty": "slightly_too_hard",
            },
        )
    )
    print(
        "\nsave_workout_feedback -> saved:",
        saved["saved"],
        "|",
        saved["feedback"]["difficulty"],
    )

    # Moving a workout is consequential: it is stored for approval, not run.
    held = output_of(
        await mc.call_tool(
            "organize_schedule",
            {"moves": [{"week_number": 1, "workout_index": 0, "new_day": "Friday"}]},
        )
    )
    print("\norganize_schedule ->", held["status"], "|", held["reason"])

    # Invalid input is rejected, by our tool or by the server's schema validation.
    missing = await mc.call_tool(
        "get_workout", {"week_number": 9, "workout_index": 0}, raise_on_error=False
    )
    print("\nUnknown workout ->", output_of(missing))
    try:
        await mc.call_tool(
            "save_workout_feedback",
            {
                "week_number": 1,
                "workout_index": 0,
                "completion_status": "completed",
                "difficulty": "hard",
            },
        )
    except Exception as error:
        print(
            "Invalid difficulty -> rejected by the server:", str(error).splitlines()[0]
        )

# A new server process only sees this record if it reached the database.
async with Client(REPWISE_CONFIG) as mc:
    reloaded = output_of(await mc.call_tool("get_timetable", {}))
print(
    "\nReloaded by a new server process:",
    reloaded["planned_workouts"][0]["feedback"]["difficulty"],
)